# Portfolio construction and valuation

**Prerequisites:** complete `01_foundations/market_data_and_curves.ipynb` and `02_pricing/pricing_fundamentals.ipynb` in this curriculum. This notebook builds a **portfolio spec** as JSON, wires **discounting** through a shared curve, and runs the **portfolio** parse → build → value → cashflow ladder pipeline.

## Concepts: portfolios as data, not objects

A **portfolio** here is a JSON **`PortfolioSpec`**: identifiers, base currency, **entities** (holders), and **positions**. Each position points at an **`instrument_spec`** (tagged JSON: `type` + `spec`)—the same shape you already use for single-instrument pricing.

The typical flow is:

1. **`parse_portfolio_spec`** — validate and canonicalize the spec string.
2. **`build_portfolio_from_spec`** — load into the engine and round-trip the spec (useful for debugging wire format).
3. **`value_portfolio`** — PV each position, convert to **base currency**, and return a **`PortfolioValuation`** JSON snapshot (per-position values, entity subtotals, **total** in base ccy).
4. **`aggregate_metrics`** — given that valuation JSON plus market and `as_of`, compute **portfolio-level metric aggregation** (summable risks, FX-aware where applicable).
5. **`aggregate_full_cashflows`** — merge contractual cashflows into a **date ladder** (`by_date`, `by_position`), collapsible to base currency with **`collapse_to_base_by_date_kind`**.

**Wire-format note:** `value_portfolio` returns **`PortfolioValuation`** only. Helpers **`portfolio_result_total_value`** and **`portfolio_result_get_metric`** expect a full **`PortfolioResult`** envelope (`valuation` + `metrics` + `meta`). The walkthrough shows both: read the total via **`PortfolioValuation.total_value`**, then compose the envelope from `value_portfolio` + `aggregate_metrics` output to exercise the helpers.

**Typed wrappers over JSON digging.** Every stage returns a typed wrapper — `PortfolioValuation`, `PortfolioResult`, `PortfolioCashflows`. Prefer their accessors to reaching into the raw payload (`json.loads(...)["total_base_ccy"]["amount"]`), and prefer engine calls to re-deriving a quantity in Python.

### Imports

Portfolio entry points live in `finstack_quant.portfolio`. The market snapshot
comes from `_shared.build_demo_market()`, the canonical example market used
across this curriculum, and the as-of date from `_shared.DEMO_AS_OF` (see
`01_foundations/market_data_and_curves.ipynb` for how that market is built).

In [1]:
import json

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, build_demo_market
from finstack_quant.portfolio import (
    PortfolioCashflows,
    PortfolioResult,
    PortfolioValuation,
    aggregate_full_cashflows,
    aggregate_metrics,
    build_portfolio_from_spec,
    parse_portfolio_spec,
    portfolio_result_get_metric,
    portfolio_result_total_value,
    value_portfolio,
)

AS_OF = DEMO_AS_OF.isoformat()
print("Imported portfolio pipeline and market helpers. as_of =", AS_OF)


Imported portfolio pipeline and market helpers. as_of = 2025-01-15


### Market context: the shared demo market

`build_demo_market()` returns a `MarketContext` carrying the curves, fixings,
prices and FX quotes that the example notebooks share. Only its `USD-OIS`
discount curve matters here — every deposit below discounts off it. Calling
`to_json()` produces the one snapshot passed into **`value_portfolio`**,
**`aggregate_metrics`**, and **`aggregate_full_cashflows`**.

In [2]:
mc = build_demo_market()
market_json = mc.to_json()
curves = json.loads(market_json).get("curves", [])
curve_ids = [c.get("id") for c in curves if isinstance(c, dict)]
print("Market OK; curve ids in snapshot:", curve_ids)

Market OK; curve ids in snapshot: ['CORP-HAZARD', 'EUR-OIS', 'USD-OIS', 'USD-SOFR-3M']


### Portfolio spec: two USD deposits

Each **position** references **`discount_curve_id`** that must exist in **`market_json`**. **`quantity`** scales the economic exposure (here `1.0` **units**).

In [3]:
portfolio = json.dumps(
    {
        "id": "credit-portfolio",
        "as_of": AS_OF,
        "base_ccy": "USD",
        "entities": {"ENTITY-1": {"id": "ENTITY-1"}},
        "positions": [
            {
                "position_id": "POS-1",
                "entity_id": "ENTITY-1",
                "instrument_id": "DEP-3M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-3M",
                        "notional": {"amount": 1_000_000.0, "currency": "USD"},
                        "start_date": AS_OF,
                        "maturity": "2025-04-15",
                        "day_count": "Act360",
                        "quote_rate": 0.045,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "POS-2",
                "entity_id": "ENTITY-1",
                "instrument_id": "DEP-6M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-6M",
                        "notional": {"amount": 2_000_000.0, "currency": "USD"},
                        "start_date": AS_OF,
                        "maturity": "2025-07-15",
                        "day_count": "Act360",
                        "quote_rate": 0.05,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
        ],
    }
)
spec_obj = json.loads(portfolio)
print("Spec id:", spec_obj["id"], "| positions:", len(spec_obj["positions"]))
print("Position ids:", [p["position_id"] for p in spec_obj["positions"]])

Spec id: credit-portfolio | positions: 2
Position ids: ['POS-1', 'POS-2']


### `parse_portfolio_spec`

Parse and canonicalize JSON. Failures raise **`ValueError`** with a Rust error string—fix the spec and retry.

In [4]:
canonical = parse_portfolio_spec(portfolio)
positions = json.loads(canonical).get("positions", [])
print("Parsed OK; canonical position count:", len(positions))
print("First instrument_id:", positions[0]["instrument_id"])

Parsed OK; canonical position count: 2
First instrument_id: DEP-3M


### `build_portfolio_from_spec`

Materialize the portfolio and return the spec after **`Portfolio::from_spec` → `to_spec`**. Use this to confirm instruments deserialize cleanly.

In [5]:
built = build_portfolio_from_spec(portfolio)
print("Round-trip prefix:", built[:200])
print("Full JSON length (chars):", len(built))

Round-trip prefix: {"id":"credit-portfolio","name":null,"base_ccy":"USD","as_of":"2025-01-15","entities":{"ENTITY-1":{"id":"ENTITY-1","name":null}},"positions":[{"position_id":"POS-1","entity_id":"ENTITY-1","instrument_
Full JSON length (chars): 393


### `value_portfolio`

Returns **`PortfolioValuation`** JSON. Wrap it with **`PortfolioValuation.from_json`**
and read **`.total_value`** / **`.base_ccy`** rather than digging into the
`total_base_ccy` money object by hand — the typed wrapper is the supported
accessor and keeps the notebook insulated from wire-format changes.

**PV sign convention (deposits).** For a **deposit** from the lender’s perspective, contractual cash flows are typically modeled as an initial **outflow** (funding the loan) and a terminal **inflow** (repayment plus interest). The engine’s **present value** is the sum of those discounted flows, so a **negative PV** means the position is economically a **loan / funding** from the holder’s viewpoint (money out today dominates discounted money in later). A **positive PV** would correspond to the opposite side of the trade (borrowing). When reading `position_values`, interpret the sign together with your **position quantity** and **book convention**.

In [6]:
val_result = value_portfolio(portfolio, market_json)
val_obj = json.loads(val_result)
print("Valuation keys:", list(val_obj.keys()))
print("Snippet:\n", json.dumps(val_obj, indent=2)[:900], "...")

valuation = PortfolioValuation.from_json(val_result)
print(f"\nTotal PV: {valuation.total_value:,.6f} {valuation.base_ccy}  (as_of {valuation.as_of})")

Valuation keys: ['as_of', 'position_values', 'total_base_ccy', 'by_entity', 'fx_collapse_policy']
Snippet:
 {
  "as_of": "2025-01-15",
  "position_values": {
    "POS-1": {
      "position_id": "POS-1",
      "entity_id": "ENTITY-1",
      "value_native": {
        "amount": "1000079.848437813",
        "currency": "USD"
      },
      "value_base": {
        "amount": "1000079.848437813",
        "currency": "USD"
      },
      "metric_scale": 1.0,
      "risk_metrics_complete": true,
      "valuation_result": {
        "schema_version": 1,
        "instrument_id": "DEP-3M",
        "as_of": "2025-01-15",
        "value": {
          "amount": "1000079.84843781325875000000",
          "currency": "USD"
        },
        "measures": {
          "theta": 122.19985535554588,
          "dv01": -24.659503114700783,
          "bucketed_dv01": -24.6595031148172,
          "bucketed_dv01::USD-OIS::10y": 0.0,
          "bucketed_dv01::USD-OIS::15y": 0.0,
          "bucketed_dv01::USD-OIS::1y

### `portfolio_result_total_value` and `portfolio_result_get_metric`

These read a full **`PortfolioResult`** envelope: **`valuation` + `metrics` + `meta`**.

`value_portfolio` gives you the `valuation` block and **`aggregate_metrics`** gives
you the `metrics` block in exactly the shape the envelope expects — so assembling a
`PortfolioResult` is a matter of putting the two engine outputs side by side. Do
**not** hand-write the metric totals or the per-entity rollup: `aggregate_metrics`
already computes them, FX-aware and entity-aware, and a hand-typed rollup silently
drifts as soon as the book has more than one entity.

(`aggregate_metrics` gets its own section below; it is called here because the
envelope needs it.)

In [7]:
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/portfolio_construction_and_valuation.json").read_text())
meta = _NOTEBOOK_DATA["meta"]

# The metrics block is engine output, not hand-assembled: `aggregate_metrics`
# returns `{"aggregated": ..., "by_position": ...}`, which is the `metrics` shape
# a PortfolioResult envelope carries.
agg = aggregate_metrics(val_result, "USD", market_json, AS_OF)
metrics_payload = json.loads(agg)

portfolio_result_json = json.dumps(
    {"valuation": val_obj, "metrics": metrics_payload, "meta": meta}
)

print("portfolio_result_total_value:", portfolio_result_total_value(portfolio_result_json))
for metric_id in ("dv01", "theta", "pv"):
    value = portfolio_result_get_metric(portfolio_result_json, metric_id)
    print(f"portfolio_result_get_metric({metric_id!r}): {value}")
print("\n('pv' is None: the aggregator emits risk measures, not the PV itself —")
print(" read the PV from PortfolioValuation.total_value or the envelope total.)")

portfolio_result_total_value: 3004606.725488827
portfolio_result_get_metric('dv01'): -124.06206883938285
portfolio_result_get_metric('theta'): 367.1329552596435
portfolio_result_get_metric('pv'): None

('pv' is None: the aggregator emits risk measures, not the PV itself —
 read the PV from PortfolioValuation.total_value or the envelope total.)


### Typed portfolio result wrapper

`PortfolioResult.from_json` wraps the same envelope used by the helper functions and exposes typed scalar accessors.

In [8]:
portfolio_result = PortfolioResult.from_json(portfolio_result_json)
print("PortfolioResult type:", PortfolioResult.__name__)
print("typed total value:", portfolio_result.total_value)
print("typed dv01 metric:", portfolio_result.get_metric("dv01"))

PortfolioResult type: PortfolioResult
typed total value: 3004606.725488827
typed dv01 metric: -124.06206883938285


### `aggregate_metrics`

Pass the **`value_portfolio`** JSON string, base currency, market JSON, and **`as_of`**.
The engine reprices each position off the supplied market and returns two blocks:

- **`aggregated`** — one entry per metric id, each with a portfolio `total` and a
  `by_entity` rollup. Summable risks (`dv01`, `theta`, and the `bucketed_dv01::…`
  key-rate ladder) add across positions; FX conversion to base currency happens here.
- **`by_position`** — the same measures per position, with the position's currency.

This is the hook for portfolio-level risk, and it is what fed the `metrics` block
of the `PortfolioResult` envelope above.

In [9]:
# `agg` was computed in the PortfolioResult section above; reuse it rather than
# repricing the book a second time.
agg_obj = json.loads(agg)

print("Aggregated metric ids:", sorted(agg_obj.get("aggregated", {}).keys()))

print("\nPortfolio totals (non-zero only):")
for metric_id, entry in agg_obj.get("aggregated", {}).items():
    if entry["total"] != 0.0:
        print(f"  {metric_id:<32} {entry['total']:>14.6f}   by_entity={entry['by_entity']}")

print("\nPer-position dv01:")
for position_id, entry in agg_obj.get("by_position", {}).items():
    print(f"  {position_id}: {entry['metrics']['dv01']:.6f} {entry['currency']}")

# The entity rollup is the engine's, not ours: totals reconcile to the per-position sum.
dv01_total = agg_obj["aggregated"]["dv01"]["total"]
dv01_sum = sum(e["metrics"]["dv01"] for e in agg_obj["by_position"].values())
print(f"\ndv01 total {dv01_total:.9f} vs sum(by_position) {dv01_sum:.9f}")

Aggregated metric ids: ['bucketed_dv01', 'bucketed_dv01::USD-OIS::10y', 'bucketed_dv01::USD-OIS::15y', 'bucketed_dv01::USD-OIS::1y', 'bucketed_dv01::USD-OIS::20y', 'bucketed_dv01::USD-OIS::2y', 'bucketed_dv01::USD-OIS::30y', 'bucketed_dv01::USD-OIS::3m', 'bucketed_dv01::USD-OIS::3y', 'bucketed_dv01::USD-OIS::5y', 'bucketed_dv01::USD-OIS::6m', 'bucketed_dv01::USD-OIS::7y', 'dv01', 'theta']

Portfolio totals (non-zero only):
  theta                                367.132955   by_entity={'ENTITY-1': 367.1329552596435}
  dv01                                -124.062069   by_entity={'ENTITY-1': -124.06206883938285}
  bucketed_dv01                       -124.062069   by_entity={'ENTITY-1': -124.06206883926644}
  bucketed_dv01::USD-OIS::1y             0.531278   by_entity={'ENTITY-1': 0.5312784345587716}
  bucketed_dv01::USD-OIS::3m           -25.568762   by_entity={'ENTITY-1': -25.568762436625548}
  bucketed_dv01::USD-OIS::6m           -99.024585   by_entity={'ENTITY-1': -99.02458483719965}



### `aggregate_full_cashflows`

Returns a typed **`PortfolioCashflows`** wrapper. Call **`to_json()`** for the full wire payload, or use its typed accessors for drill-downs. Amounts serialize as decimal strings on the wire.

In [10]:
cfs = aggregate_full_cashflows(portfolio, market_json)
cf_obj = json.loads(cfs.to_json())
print(json.dumps(cf_obj, indent=2)[:1400])
print("... [truncated if long] ...")
print("Ladder dates:", sorted(cf_obj.get("by_date", {}).keys()))

{
  "events": [
    {
      "position_id": "POS-1",
      "instrument_id": "DEP-3M",
      "instrument_type": "Deposit",
      "date": "2025-01-15",
      "amount": {
        "amount": "-1000000",
        "currency": "USD"
      },
      "kind": "Notional",
      "reset_date": null,
      "accrual_factor": 0.0,
      "rate": null
    },
    {
      "position_id": "POS-2",
      "instrument_id": "DEP-6M",
      "instrument_type": "Deposit",
      "date": "2025-01-15",
      "amount": {
        "amount": "-2000000",
        "currency": "USD"
      },
      "kind": "Notional",
      "reset_date": null,
      "accrual_factor": 0.0,
      "rate": null
    },
    {
      "position_id": "POS-1",
      "instrument_id": "DEP-3M",
      "instrument_type": "Deposit",
      "date": "2025-04-15",
      "amount": {
        "amount": "1011250",
        "currency": "USD"
      },
      "kind": "Fixed",
      "reset_date": null,
      "accrual_factor": 0.25,
      "rate": 0.045
    },
    {
      "posi

### Typed cashflow ladder wrapper

`aggregate_full_cashflows` returns `PortfolioCashflows`, not raw JSON; serialize only when crossing a wire boundary.

In [11]:
print("PortfolioCashflows type:", PortfolioCashflows.__name__)
print("cfs is PortfolioCashflows:", isinstance(cfs, PortfolioCashflows))
print("positions:", cfs.num_positions, "| issues:", cfs.num_issues)
print("events json bytes:", len(cfs.events_json()))

PortfolioCashflows type: PortfolioCashflows
cfs is PortfolioCashflows: True
positions: <built-in method num_positions of finstack_quant.portfolio.PortfolioCashflows object at 0x1157c1d70> | issues: <built-in method num_issues of finstack_quant.portfolio.PortfolioCashflows object at 0x1157c1d70>
events json bytes: 876


## Mini-example: three deposits (3M, 6M, 1Y)

Same market and `as_of`. Different notionals and quoted rates. We print **total PV**,
the aggregated **dv01** ladder, and a readable **cashflow ladder** sorted by date.

For the ladder, use **`PortfolioCashflows.collapse_to_base_by_date_kind`** rather
than summing the `by_date` amounts in Python. The engine collapses multi-currency
flows to base currency with spot-equivalent FX at each payment date; a Python
`sum(float(...))` over the raw `by_date` map only happens to work while every flow
is already in the base currency, and silently adds unlike currencies the moment it
is not.

In [12]:
def deposit_position(
    position_id: str,
    instrument_id: str,
    notional: float,
    quote_rate: float,
    maturity: str,
) -> dict:
    return {
        "position_id": position_id,
        "entity_id": "ENTITY-1",
        "instrument_id": instrument_id,
        "instrument_spec": {
            "type": "deposit",
            "spec": {
                "id": instrument_id,
                "notional": {"amount": notional, "currency": "USD"},
                "start_date": AS_OF,
                "maturity": maturity,
                "day_count": "Act360",
                "quote_rate": quote_rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
    }


three_deposit_portfolio = json.dumps(
    {
        "id": "rates-ladder-book",
        "as_of": AS_OF,
        "base_ccy": "USD",
        "entities": {"ENTITY-1": {"id": "ENTITY-1"}},
        "positions": [
            deposit_position("L-3M", "DEP-3M-ME", 1_000_000.0, 0.045, "2025-04-15"),
            deposit_position("L-6M", "DEP-6M-ME", 2_000_000.0, 0.050, "2025-07-15"),
            deposit_position("L-1Y", "DEP-1Y-ME", 1_500_000.0, 0.055, "2026-01-15"),
        ],
    }
)

mini_val = value_portfolio(three_deposit_portfolio, market_json)
mini_valuation = PortfolioValuation.from_json(mini_val)
print(f"Three-deposit total PV: {mini_valuation.total_value:,.6f} {mini_valuation.base_ccy}")

mini_agg = json.loads(aggregate_metrics(mini_val, "USD", market_json, AS_OF))
print(f"Portfolio dv01:        {mini_agg['aggregated']['dv01']['total']:,.6f}")
print("Per-position dv01:")
for position_id, entry in mini_agg["by_position"].items():
    print(f"  {position_id}: {entry['metrics']['dv01']:>12.6f} {entry['currency']}")

# Base-currency ladder straight from the engine: (date, CFKind) -> Money.
mini_cfs = aggregate_full_cashflows(three_deposit_portfolio, market_json)
ladder = json.loads(mini_cfs.collapse_to_base_by_date_kind(market_json, "USD", AS_OF))

print("\nCashflow ladder (base USD, by date and cashflow kind):")
for cf_date in sorted(ladder):
    for kind, money in sorted(ladder[cf_date].items()):
        print(f"  {cf_date}  {kind:<10} {float(money['amount']):>15,.2f} {money['currency']}")
print("Done.")

Three-deposit total PV: 4,516,988.496322 USD
Portfolio dv01:        -275.300246
Per-position dv01:
  L-3M:   -24.659503 USD
  L-6M:   -99.402566 USD
  L-1Y:  -151.238177 USD

Cashflow ladder (base USD, by date and cashflow kind):
  2025-01-15  Notional     -4,500,000.00 USD
  2025-04-15  Fixed         1,011,250.00 USD
  2025-07-15  Fixed         2,050,277.78 USD
  2026-01-15  Fixed         1,583,645.83 USD
Done.


## Performance (TWRR / MWRR) and Brinson attribution

Portfolio-level returns and attribution are available via lightweight JSON helpers. These operate on simple period/cashflow or sector-return records and do not require a full `Portfolio` object.


In [13]:
from finstack_quant.portfolio import (
    brinson_fachler,
    carino_link,
    mwr_xirr,
    twrr_linked,
)

# Link sub-period returns (TWRR linking)
linked = twrr_linked(json.dumps([0.03, 0.025, 0.04]), 1.0)
print("twrr_linked (annualized):", linked)

# MWR (XIRR)
cf = json.dumps([
    {"date": "2025-01-01", "amount": -1_000_000.0},
    {"date": "2025-06-30", "amount": 50_000.0},
    {"date": "2026-01-01", "amount": 1_080_000.0},
])
x = mwr_xirr(cf)
print("mwr_xirr:", round(x, 6))

# Brinson-Fachler (single period)
sector_rows = [
    {"sector": "Tech", "portfolio_weight": 0.6, "benchmark_weight": 0.5, "portfolio_return": 0.04, "benchmark_return": 0.03},
    {"sector": "Fin",  "portfolio_weight": 0.4, "benchmark_weight": 0.5, "portfolio_return": 0.01, "benchmark_return": 0.015},
]
sectors = json.dumps(sector_rows)
bf = json.loads(brinson_fachler(sectors))
print("brinson_fachler excess:", round(bf.get("total_excess_return", 0), 6))

# Carino link (multi-period)
periods = json.dumps([sector_rows, sector_rows])  # toy reuse
cl = json.loads(carino_link(periods))
print("carino_link excess:", round(cl.get("total_excess_return", 0), 6))

twrr_linked (annualized): {"cumulative":0.09797999999999996,"annualised":0.09797999999999996,"num_periods":3}
mwr_xirr: 0.133273
brinson_fachler excess: 0.0055
carino_link excess: 0


## Takeaways

- A **portfolio** is **`PortfolioSpec` JSON**: entities + positions with embedded **`instrument_spec`** blocks.
- **`_shared.build_demo_market().to_json()`** is the shared **market snapshot** for value, metrics, and cashflows; **`_shared.DEMO_AS_OF`** is the matching as-of date.
- **`value_portfolio`** produces **`PortfolioValuation`** JSON; read the total with **`PortfolioValuation.from_json(...).total_value`**, not by digging out `total_base_ccy.amount`.
- **`aggregate_metrics`** returns the portfolio's summed risks (`dv01`, `theta`, the `bucketed_dv01::…` key-rate ladder) with a `by_entity` rollup and a `by_position` breakdown.
- **`portfolio_result_*`** helpers expect a **`PortfolioResult`** envelope (**`valuation` + `metrics` + `meta`**) — compose it from `value_portfolio` and `aggregate_metrics` output rather than hand-writing the metric totals.
- **`aggregate_full_cashflows`** gives a **date-indexed ladder**; **`collapse_to_base_by_date_kind`** converts it to one base-currency `(date, kind)` ladder using engine FX, which is the currency-safe way to total a ladder.